## imports 

In [1]:
import logging
import os

import lightning

from openretina.data_io.base import MoviesTrainTestSplit, ResponsesTrainTestSplit, compute_data_info
from openretina.data_io.closedloopdensenoise.responses import load_all_responses
from openretina.data_io.closedloopdensenoise.stimuli import load_all_stimuli
from openretina.data_io.closedloopdensenoise.constants import STIMULUS_SHAPE


from openretina.data_io.base_dataloader import multiple_movies_dataloaders
from openretina.data_io.cyclers import LongCycler, ShortCycler
from openretina.models.core_readout import CoreReadout
from openretina.utils.file_utils import get_cache_directory, get_local_file_path
from openretina.utils.h5_handling import load_dataset_from_h5, load_h5_into_dict
from openretina.utils.misc import CustomPrettyPrinter

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)  # to display logs in jupyter notebooks

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
pp = CustomPrettyPrinter(indent=4, max_lines=40)

## loading necessary tables from data base

In [2]:
import numpy as np
import h5py

## data loaders from TrainTestSplit objects

In [84]:
stimuli = load_all_stimuli()
responses = load_all_responses()
dataloaders = multiple_movies_dataloaders(neuron_data_dictionary=responses, 
                                          movies_dictionary=stimuli,
                                          clip_length = STIMULUS_SHAPE[0] * 2,
                                          train_chunk_size = STIMULUS_SHAPE[0] * 2 , # the size of the chunks to be used for training. The sampler also shuffles between chunks
                                          batch_size = 9,
                                          )

pp.pprint(dataloaders)

Creating movie dataloaders:   0%|          | 0/1 [00:00<?, ?it/s]

defaultdict(<class 'dict'>,
            {   'test': {   'iter0': torch.utils.data.DataLoader(Dataset: MovieDataSet with 101 neuron responses to a movie of shape [1, 300, 20, 15].)},
                'train': {   'iter0': torch.utils.data.DataLoader(Dataset: MovieDataSet with 101 neuron responses to a movie of shape [1, 700, 20, 15].)},
                'validation': {   'iter0': torch.utils.data.DataLoader(Dataset: MovieDataSet with 101 neuron responses to a movie of shape [1, 500, 20, 15].)}})


## model 

In [85]:
data_info = compute_data_info(neuron_data_dictionary=responses, movies_dictionary=stimuli)

# Display the data info
pp.pprint(data_info)

{   'input_shape': (1, 20, 15),
    'movie_norm_dict': {   'iter0': {   'norm_mean': 0.5003694295883179,
                                        'norm_std': 0.500000536441803}},
    'n_neurons_dict': {'iter0': 101},
    'sessions_kwargs': {'iter0': {}}}


In [86]:
model = CoreReadout(
    in_shape=(1, 100, 20, 15),  # Note that data_info does not include time, we add a dummy time dimension here.
    hidden_channels=[32, 64], # Number of filters for each convolutional layer
    temporal_kernel_sizes=[3, 3], # Kernel size for temporal convolutions for each convolutional layer
    spatial_kernel_sizes=[7, 7], # Kernel size for spatial convolutions for each convolutional layer
    n_neurons_dict=data_info["n_neurons_dict"],
)

2025-04-28 12:01:04,494 - INFO - in_shape_readout=torch.Size([64, 68, 14, 9])


In [87]:
train_loader = LongCycler(dataloaders["train"])
val_loader = ShortCycler(dataloaders["validation"])
test_loader = ShortCycler(dataloaders["test"])

In [88]:
trainer = lightning.Trainer(max_epochs=1)

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                        | Params | Mode 
-------------------------------------------------------------------------
0 | core             | SimpleCoreWrapper           | 106 K  | train
1 | readout          | MultiGaussianReadoutWrapper | 7.0 K  | train
2 | loss             | PoissonLoss3d               | 0      | train
3 | correlation_loss | CorrelationLoss3d           | 0      | train
-------------------------------------------------------------------------
113 K     Trainable params
0         Non-trainable params
113 K     Total params
0.453     Total estimated model params size (MB)
18        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.
